In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
import os
import pickle

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2, combine_pvalues
from sklearn.model_selection import StratifiedShuffleSplit
from statsmodels.stats.multitest import multipletests

def hosmer_lemeshow_test(observed, predicted_probs, n_groups=10):
    df_hl = pd.DataFrame({
        'observed': observed,
        'predicted_probability': predicted_probs
    })
    df_hl['group'] = pd.qcut(df_hl['predicted_probability'], n_groups, duplicates='drop')

    hl_table = df_hl.groupby('group').apply(
        lambda x: pd.Series({
            'observed': x['observed'].sum(),
            'expected': x['predicted_probability'].sum(),
            'total': len(x)
        })
    )

    hl_table['observed_neg'] = hl_table['total'] - hl_table['observed']
    hl_table['expected_neg'] = hl_table['total'] - hl_table['expected']

    hl_statistic = (((hl_table['observed'] - hl_table['expected']) ** 2) / hl_table['expected'] +
                    ((hl_table['observed_neg'] - hl_table['expected_neg']) ** 2) / hl_table['expected_neg']).sum()

    df = n_groups - 2
    p_value = 1 - chi2.cdf(hl_statistic, df)
    return hl_statistic, p_value

def hosmer_lemeshow(observed, predicted_probs, n_repeats=100, n_groups=10, test_size=0.1, random_state=None):
    hl_stats = []
    p_values = []

    sss = StratifiedShuffleSplit(n_splits=n_repeats, test_size=test_size, random_state=random_state)

    observed = np.array(observed)
    predicted_probs = np.array(predicted_probs)

    for train_index, test_index in sss.split(predicted_probs, observed):
        y_test = observed[test_index]
        prob_test = predicted_probs[test_index]
        try:
            hl_stat, p_val = hosmer_lemeshow_test(y_test, prob_test, n_groups)
            hl_stats.append(hl_stat)
            p_values.append(p_val)
        except Exception:
            continue

    hl_stat_mean = np.mean(hl_stats)
    fisher_stat, fisher_p_value = combine_pvalues(p_values, method='fisher')

    df = n_groups - 2
    p_from_mean_stat = 1 - chi2.cdf(hl_stat_mean, df)

    return {
        'hl_stat_mean': hl_stat_mean,
        'fisher_p_value': fisher_p_value,
        'p_from_mean_stat': p_from_mean_stat,
        'all_hl_stats': hl_stats,
        'all_p_values': p_values
    }

In [ ]:
warnings.filterwarnings('ignore')
np.random.seed(42)

# Load data

## BAGs

In [ ]:
df =  pd.read_parquet('../data/main_data_ses.parquet')
result_dict = {}

In [ ]:
vars_list=['wellbeing_score',
 'institutional_values_score',
 'social_life_values_score',
 'agency_score',
 'all_score']

## Todos

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = df[['GAP_corrected', feature, 'SES_wo_w']].dropna(axis=0).copy()


    if feature == "trust_your_family":
        temp_df = temp_df[temp_df.trust_your_family <= 1.5].reset_index(drop=True)


    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")

    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature, 'SES_wo_w']].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)





In [ ]:
result_dict['mediana_global'] = df_or_summary
result_dict['mediana_global-hl'] = df_hl_summary
result_dict['mediana_global-or_list'] = df_or_wide

## Separando por continente

### LatinAmerica

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = df[['GAP_corrected', feature, 'continent', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continent == 'LatinAmerica']
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")


    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)



In [ ]:
result_dict['latam'] = df_or_summary
result_dict['latam-hl'] = df_hl_summary
result_dict['latam-or_list'] = df_or_wide

### Latin America by Income

In [ ]:
lm_c = df.copy()
lm_c.continent.unique()

In [ ]:
lm_c=lm_c.loc[lm_c['continent']=="LatinAmerica"]

In [ ]:
lm_c.wvs_ccode.unique()

In [ ]:
lm_c['continent'] = np.where(lm_c['wvs_ccode'].isin(['Brasil', 'Mexico', 'Colombia']), 'MHI', lm_c['continent'])
lm_c['continent'] = np.where(lm_c['wvs_ccode'].isin(['Chile', 'Ecuador', 'Uruguay']), 'MLI', lm_c['continent'])

In [ ]:
lm_c.continent.unique()

In [ ]:
def remove_outliers_iqr(df, col, group_col, threshold=1.5):
    import pandas as pd
    import numpy as np
    outlier_idx = []
    for g in df[group_col].unique():
        vals = df.loc[df[group_col]==g, col]
        Q1 = np.percentile(vals, 25)
        Q3 = np.percentile(vals, 75)
        IQR = Q3 - Q1
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR

        # índices de outliers
        idx = df.loc[(df[group_col]==g) & ((df[col]<lower) | (df[col]>upper))].index.tolist()
        outlier_idx.extend(idx)

    # devolver df sin outliers
    return df.drop(index=outlier_idx).reset_index(drop=True)

# ejemplo con tu df
lm_c_1 = remove_outliers_iqr(lm_c, col="GAP_corrected", group_col="wvs_ccode", threshold=1.5)
lm_c_2 = remove_outliers_iqr(lm_c, col="GAP_corrected", group_col="wvs_ccode", threshold=1.75)
lm_c_3 = remove_outliers_iqr(lm_c, col="GAP_corrected", group_col="wvs_ccode", threshold=1.25)


In [ ]:
# lm_c.loc[lm_c['wvs_ccode'].isin(['Chile', 'Uruguay']), ['wvs_ccode', 'continent']]
lm_c=lm_c.reset_index(drop=True)

### Latin America Middle-High income

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = lm_c[['GAP_corrected', feature, 'continent', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continent.isin(['MHI'])]
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")


    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)


In [ ]:
result_dict['latam_MHI'] = df_or_summary
result_dict['latam-hl_MHI'] = df_hl_summary
result_dict['latam-or_list_MHI'] = df_or_wide

### Latin America Middle-Low income

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = lm_c[['GAP_corrected', feature, 'continent', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continent == 'MLI']
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")


    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)



In [ ]:
result_dict['latam_MLI'] = df_or_summary
result_dict['latam-hl_MLI'] = df_hl_summary
result_dict['latam-or_list_MLI'] = df_or_wide

### Europe

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = df[['GAP_corrected', feature, 'continent', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continent == 'Europe']
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")


    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)


In [ ]:
result_dict['europa'] = df_or_summary
result_dict['europa-hl'] = df_hl_summary
result_dict['europa-or_list'] = df_or_wide

### West europe

In [ ]:
west_europe = ['France', 'Germany', 'Italy', 'Netherlands', 'Spain', 'Switzerland', 'Cyprus']
east_europe = ['Estonia', 'Greece', 'Poland', 'Romania', 'Slovak Republic', 'Hungary', 'Slovenia']

df['continentEurope'] = np.select(
    [
        df['wvs_ccode'].isin(west_europe),
        df['wvs_ccode'].isin(east_europe),
    ],
    ['west_europe', 'east_europe'],
    default='non_europe'
)

df['continentEurope'] = df['continentEurope'].astype('category')

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = df[['GAP_corrected', feature, 'continentEurope', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continentEurope == 'west_europe']
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")

    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)


In [ ]:
result_dict['west_europe'] = df_or_summary
result_dict['west_europe-hl'] = df_hl_summary
result_dict['west_europe-or_list'] = df_or_wide

### East europe

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from scipy.stats import chi2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


n_splits   = 20
test_size  = 0.10
hl_groups  = [20, 50, 100]
z_mult     = 2.46


rows = []

for feature in vars_list[-5:]:

    temp_df = df[['GAP_corrected', feature, 'continentEurope', 'wvs_ccode']].dropna(axis=0).copy()

    temp_df = temp_df[temp_df.continentEurope == 'east_europe']
    temp_df.reset_index(inplace=True, drop = True)

    temp_df['GAP_bin'] = np.where(
        temp_df['GAP_corrected'] > 0, 1,
        np.where(temp_df['GAP_corrected'] <= 0, 0, np.nan)
    )
    temp_df = temp_df.dropna(subset=['GAP_bin'])
    sample_size = temp_df.shape[0]
    print(f"sample size for {feature}: {sample_size}")

    y_all = temp_df['GAP_bin'].astype(int)

    X_all = temp_df[[feature]].copy()


    for iter_idx, seed in enumerate(range(n_splits), start=1):
        try:

            stratifier = y_all if y_all.nunique() == 2 else None
            X_train, X_test, y_train, y_test = train_test_split(
                X_all, y_all, test_size=test_size, random_state=seed, stratify=stratifier
            )


            X_train = X_train.copy()
            X_test  = X_test.copy()
            X_train['intercept'] = 1.0
            X_test['intercept']  = 1.0
            X_train = X_train.astype(float)
            X_test  = X_test.astype(float)


            model = sm.Logit(y_train, X_train).fit(disp=0)


            params = model.params
            ci     = model.conf_int()
            OR     = float(np.exp(params[feature]))
            LCL    = float(np.exp(ci.loc[feature, 0]))
            UCL    = float(np.exp(ci.loc[feature, 1]))
            p_val   = model.pvalues[feature]
            z_stat   = model.tvalues[feature]


            y_test_pred = model.predict(X_test)

            hl_stats = {}
            for ng in hl_groups:
                try:

                    result_hl = hosmer_lemeshow(
                        observed=y_test,
                        predicted_probs=y_test_pred,
                        n_repeats=100,
                        n_groups=ng,
                        test_size=test_size,
                        random_state=seed
                    )
                    hl_stat_mean = result_hl.get('hl_stat_mean', np.nan)
                    hl_pval      = 1 - chi2.cdf(hl_stat_mean, df=ng - 2) if np.isfinite(hl_stat_mean) else np.nan
                except Exception:
                    hl_stat_mean, hl_pval = np.nan, np.nan

                hl_stats[f'HL_stat_ng{ng}'] = hl_stat_mean
                hl_stats[f'HL_pval_ng{ng}'] = hl_pval


            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': OR,
                '2.5%': LCL,
                '97.5%': UCL,
                'z': z_stat,
                'P>|z|': p_val,
                'logOR': np.log(OR),
                **hl_stats
            })

        except Exception as e:

            rows.append({
                'Feature': feature,
                'iter': iter_idx,
                'seed': seed,
                'OR': np.nan,
                '2.5%': np.nan,
                '97.5%': np.nan,
                'z': np.nan,
                'P>|z|': np.nan,
                'logOR': np.nan,
                'error': str(e)[:200]
            })


df_or_iters = pd.DataFrame(rows).reset_index(drop=True)

def summarize_logOR(g, z=z_mult):
    logor = g['logOR'].replace([np.inf, -np.inf], np.nan).dropna()
    pval = g['P>|z|'].replace([np.inf, -np.inf], np.nan).dropna()
    z_val = g['z'].replace([np.inf, -np.inf], np.nan).dropna()
    iters_ok = int(logor.notna().sum())
    iters_tot = int(len(g))
    if iters_ok == 0:
        return pd.Series({
            'OR_geo_mean': np.nan,
            'OR_LCL': np.nan,
            'OR_UCL': np.nan,
            'sd_logOR': np.nan,
            'z': np.nan,
            'P>|z|': np.nan,
            'iters_ok': iters_ok,
            'iters_tot': iters_tot
        })
    m  = float(np.nanmean(logor))
    sd = float(np.nanstd(logor, ddof=1))
    OR_geo_mean = float(np.exp(m))
    LCL = float(np.exp(m - z * sd))
    UCL = float(np.exp(m + z * sd))
    pval = float(np.nanmean(pval))
    z_val = float(np.nanmean(z_val))
    return pd.Series({
        'OR_geo_mean': OR_geo_mean,
        'OR_LCL': LCL,
        'OR_UCL': UCL,
        'sd_logOR': sd,
        'z': z_val,
        'P>|z|': pval,
        'iters_ok': iters_ok,
        'iters_tot': iters_tot
    })

df_or_summary = (
    df_or_iters
      .groupby('Feature')
      .apply(summarize_logOR)
      .reset_index()
      .sort_values('OR_geo_mean')
      .reset_index(drop=True)
)


plt.figure(figsize=(5, 6))

x  = df_or_summary['OR_geo_mean'].values
xl = x - df_or_summary['OR_LCL'].values
xu = df_or_summary['OR_UCL'].values - x

plt.errorbar(
    x, df_or_summary['Feature'],
    xerr=[xl, xu],
    fmt='o', ecolor='black', elinewidth=2, capsize=5
)

for _, row in df_or_summary.iterrows():
    label = f"{row['OR_geo_mean']:.2f} [{row['OR_LCL']:.2f}-{row['OR_UCL']:.2f}]"
    plt.text(row['OR_geo_mean'] * 1.05, row['Feature'], label, va='center', ha='left', fontsize=9)

plt.axvline(x=1, color='red', linestyle='--', linewidth=2)
plt.title('Odds Ratios across repeated splits (geom. mean & CI via 2.46·sd(logOR))')
plt.xlabel('Odds Ratio')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


df_or_wide = (
    df_or_iters
      .pivot(index='Feature', columns='iter', values='OR')
      .sort_index()
)

# Inspect quickly
df_or_iters.filter(regex=r'^HL_').head()


hl_long_parts = []
for ng in hl_groups:
    part = (
        df_or_iters[['Feature', 'iter', f'HL_stat_ng{ng}', f'HL_pval_ng{ng}']]
          .rename(columns={f'HL_stat_ng{ng}':'HL_stat', f'HL_pval_ng{ng}':'HL_pval'})
          .copy()
    )
    part['n_groups'] = ng
    hl_long_parts.append(part)

hl_long = pd.concat(hl_long_parts, ignore_index=True)


df_hl_summary = (
    hl_long
      .groupby(['Feature', 'n_groups'], as_index=False)
      .agg(mean_stat   = ('HL_stat', 'mean'),
           sd_stat     = ('HL_stat', 'std'),
           median_stat = ('HL_stat', 'median'),
           mean_pval   = ('HL_pval', 'mean'),
           prop_reject_0_05 = ('HL_pval', lambda s: float(np.mean(s < 0.05))))
)

df_hl_summary['pval_from_mean_stat'] = 1 - chi2.cdf(
    df_hl_summary['mean_stat'],
    df_hl_summary['n_groups'] - 2
)


In [ ]:
result_dict['east_europe'] = df_or_summary
result_dict['east_europe-hl'] = df_hl_summary
result_dict['east_europe-or_list'] = df_or_wide

In [ ]:
import pickle

with open("../results/OR_Main_SES.pkl", "wb") as f:
    pickle.dump(result_dict, f)